In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.visualisation.spatial_visualisation import SpatialVisualisation
from quinex_application.postprocessing.data_loading import load_property_data
from quinex_application.postprocessing.data_processing import (
    describe_corpus, save_text_availability_split,
    load_text_availability_split, merge_text_availability_splits,
)
from quinex_application.visualisation.plot_utils import show_figure

# True: temporal/spatial plots use only records from full-text publications
# (incl. IQR filter, aggregates, regression and all statistics derived from the plots)
FULL_TEXT_ONLY = False
CORPUS = "full_text" if FULL_TEXT_ONLY else "all"  # suffix for saved files

TECHNOLOGIES_NAN_PATH = "../data/visualisation_files/df_technologies_nan.parquet"
UNIT = "http://qudt.org/vocab/unit/"
# Contributing DOIs per figure, to merge counts of panels combined from several notebooks
CONTRIBUTING_DOIS_DIR = "../data/figures/contributing_dois"

**Corpus:** set `FULL_TEXT_ONLY` in the first cell to `False` (default: full text and abstract-only publications) or `True` (only records from full-text publications). It is passed to the plot functions as `full_text_only`, which restrict every dataset before any filtering, so all statistics (IQR filter, aggregates, regression slope, MAE, data tables) refer to the selected corpus. Every figure states the corpus and the number of contributing publications below the plot, and saved files get the suffix `_all` / `_full_text`.

In [ ]:
visualisation = SpatialVisualisation()

### **Import Results**

In [ ]:
cost_lcoh = pd.read_excel("../data/visualisation_files/unit_converted/df_technologies_lcoh.xlsx")

In [ ]:
cost_lcoh["entity_mapping"].value_counts().head(20)

In [ ]:
cost_lcoh[cost_lcoh["entity_base_mapping"] == "Hydrogen"]["property"].value_counts().tail(33)

In [ ]:
cost_lcoh[cost_lcoh["entity_base_mapping"] == "Hydrogen"]["property"].value_counts().head(20)

#### **LCOE**

In [ ]:
cost_lcoe = pd.read_excel("../data/visualisation_files/unit_converted/df_technologies_lcoe2024.xlsx")
cost_lcoe = cost_lcoe[cost_lcoe["entity_base_mapping"].isin(['PV'])]

In [ ]:
cost_lcoe = visualisation.convert_spatial(cost_lcoe, spatial_col="spatial_scope")
print(cost_lcoe["entity_base_mapping"].value_counts())

In [ ]:
cost_lcoe = cost_lcoe[cost_lcoe["value_converted"] < 510]
len(cost_lcoe)

### **Visualize Results**

In [ ]:
datasets = [
    {
    'df': cost_lcoe,
    'value_column': 'value_converted',
    'title': 'c) PV',
    'plot_mode': 'choropleth',
    'min_datapoints': 3,
    },
    {
    'df': cost_lcoe,
    'value_column': 'value_converted',
    'title': 'd) PV',
    'plot_mode': 'points',
    'spatial_extent': [-20, 60, 5, 40],
    },
    {
    'df': cost_lcoe,
    'value_column': 'value_converted',
    'title': 'c) PV',
    'plot_mode': 'size',
    'ghi_data_path': "../data/visualisation_files/ghi/country_GHI_mean.xlsx",
    'highlight_countries': ['CHN', 'USA', 'SAU', 'MOZ', 'CAN'],
    'x_range': [3,7],
    #'min_datapoints': 5,
    #'show_regression': True,
    },
]

fig, data_tables = visualisation.plot_multiple_datasets_on_world(
    datasets,
    cmap="viridis_r",
    colorbar_title="LCOE [$/MWh]",
    figsize=(26, 12),
    jitter_strength=0,
    default_point_size=150,
    font_size=30,
    #color_scale_range=(0, 520),
    #colorbar_ticks=[0, 100, 200, 300, 400, 500],
    iqr_lower_quantile=0.25,
    iqr_upper_quantile=0.75,
    iqr_multiplier=3,
    full_text_only=FULL_TEXT_ONLY,
)

show_figure(fig)
# show_figure(fig, save_path=f"../data/figures/results/spatial/lcoe_pv_new2_{CORPUS}.pdf")

#### **Meta-regression: Drivers of PV LCOE**

The country means above mix studies with different years, system scales, technological scopes, financing assumptions and study designs. A linear mixed model on log(LCOE) with a random intercept per publication (records of one paper are not independent) estimates the effect of each covariate. Coefficients are reported as % change of LCOE.

- `year_c`: year the value refers to (temporal scope, else publication year) − 2020
- `projection`: year the value refers to lies after the publication year (e.g. 2030, 2050)
- `ghi`: country mean GHI [kWh/m²/day]
- `application`: off-grid / rooftop-residential / utility-scale / unspecified (regex on extracted entity)
- `non_pv`: CSP / solar thermal / PVT mapped to PV
- `discount_rate`, `dr_missing`: discount/interest rate [%] reported in the same paper (median-imputed) and missing flag
- `real_world`: real system vs. model system (Quinex classification)
- `cited`: value cited from another source (Quinex reference/classification)
- `full_text`: extracted from full text vs. abstract only

In [ ]:
from quinex_application.postprocessing.meta_regression import (
    build_regression_data, discount_rate_per_doi, fit_mixed_model,
    effect_table, compare_country_effects, country_lr_test,
    icc, country_variance_components,
)

# All quantities of the contributing publications, to find reported discount rates
lcoe_dois = list(cost_lcoe["DOI"].dropna().unique())
quantities_lcoe_dois = pd.read_parquet(
    TECHNOLOGIES_NAN_PATH,
    columns=["DOI", "Property", "Value Normalized", "Unit Normalized"],
    filters=[("DOI", "in", lcoe_dois)],
)
discount_rates = discount_rate_per_doi(quantities_lcoe_dois)

ghi_data = visualisation._load_ghi_data("../data/visualisation_files/ghi/country_GHI_mean.xlsx")
reg_data = build_regression_data(
    cost_lcoe, ghi_data, discount_rates, min_country_records=5, min_country_publications=5,
)
reg_data = reg_data.dropna(subset=["ghi"])
print(f"{len(reg_data)} records from {reg_data['DOI'].nunique()} publications, "
      f"discount rate reported in {discount_rates.size} publications")

In [ ]:
COVARIATES = (
    "year_c + projection + C(application, Treatment('unspecified')) + non_pv"
    " + discount_rate + dr_missing + real_world + cited + full_text"
)

lcoe_model = fit_mixed_model(reg_data, f"log_lcoe ~ ghi + {COVARIATES}")
print(lcoe_model.summary())
print(f"ICC (share of residual variance between publications): {icc(lcoe_model):.2f}")
effect_table(lcoe_model)

**Country effects before and after adjustment.** Countries with ≥ 5 records from ≥ 5 publications get their own coefficient vs. the pooled rest (`Other`). GHI is left out here because it is constant within a country. *Unadjusted* already includes the random intercept per publication, so it is not the arithmetic mean of the map. p-values are Holm-corrected for the number of country comparisons; the likelihood-ratio test (ML fit) is the global test whether country explains LCOE beyond the covariates.

**Variance decomposition.** Country as random effect with publications nested in countries. The drop of the country variance from the model without covariates is the share of the between-country spread explained by study composition (and, in the last model, by irradiation).

In [ ]:
country_effects = compare_country_effects(reg_data, COVARIATES)
for label, covariates in [("unadjusted", None), ("adjusted", COVARIATES)]:
    lr = country_lr_test(reg_data, covariates)
    print(f"Country effect ({label}): LR = {lr['LR']:.1f}, df = {lr['df']}, p = {lr['p']:.4f}")
display(country_effects.round(3))

variance_components = country_variance_components(reg_data, {
    "no covariates": None,
    "study composition": COVARIATES,
    "study composition + GHI": f"{COVARIATES} + ghi",
})
variance_components.round(3)
# country_effects.to_excel(f"../data/figures/results/spatial/lcoe_pv_country_effects_{CORPUS}.xlsx", index=False)

**Dumbbell plot.** Country effects without (grey) and with (blue, 95% CI) covariates, relative to the pooled `Other` group; countries in bold are significant after Holm correction.

In [ ]:
from quinex_application.visualisation.meta_regression_visualisation import plot_country_effects_dumbbell
from quinex_application.postprocessing.country_codes import COUNTRY_NAME_VARIANTS

# First (shortest common) name per ISO3 code for the axis labels
iso3_to_name = {}
for name, iso3 in COUNTRY_NAME_VARIANTS.items():
    iso3_to_name.setdefault(iso3, name)

fig = plot_country_effects_dumbbell(
    country_effects,
    reference_label="other countries",
    country_names=iso3_to_name,
    corpus_note=describe_corpus(CORPUS, df=reg_data, source_col="source_url"),
)
show_figure(fig)
# show_figure(fig, save_path=f"../data/figures/results/spatial/lcoe_pv_country_effects_dumbbell_{CORPUS}.pdf")

### **Number of Scenarios**

In [ ]:
# Only load this property (the full file needs ~7.5 GB RAM)
number_scenarios = load_property_data(
    TECHNOLOGIES_NAN_PATH, "Number of Scenarios", unit_uris=UNIT + "UNITLESS"
)
number_scenarios["unit_normalized"].apply(lambda units: units[0]["unit_uri"]).value_counts()

In [ ]:
number_scenarios = visualisation.convert_spatial(number_scenarios, spatial_col="spatial_scope")
print(number_scenarios["property_base_mapping"].value_counts())

In [ ]:
datasets = [
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'b',
    'plot_mode': 'choropleth',
    'min_datapoints': 3,
    },
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'c',
    'plot_mode': 'points',
    'spatial_extent': [-117, -92, 18, 43],
    },
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'd',
    'plot_mode': 'points',
    'spatial_extent': [118.0, 123.5, 20.0, 41.0],
    },
    # {
    # 'df': number_scenarios,
    # 'value_column': 'value_normalized',
    # 'title': 'e',
    # 'plot_mode': 'points',
    # 'spatial_extent': [0, 15, 41.0, 51],
    # },
    {
    'df': number_scenarios,
    'value_column': 'value_normalized',
    'title': 'e',
    'plot_mode': 'points',
    'spatial_extent': [38, 61, 20, 38],
    },
]

fig, data_tables = visualisation.plot_multiple_datasets_on_world(
    datasets,
    colorbar_title="Number of Scenarios",
    figsize=(26, 12),
    jitter_strength=0,
    default_point_size=300,
    font_size=30,
    color_scale_range=(2, 7),
    colorbar_ticks=[3, 5, 7],
    iqr_lower_quantile=0.25,
    iqr_upper_quantile=0.75,
    iqr_multiplier=3,
    full_text_only=FULL_TEXT_ONLY,
)

show_figure(fig)
save_text_availability_split(
    visualisation.last_contributing_split,
    f"{CONTRIBUTING_DOIS_DIR}/number_scenarios_{CORPUS}.json",
)
# show_figure(fig, save_path=f"../data/figures/results/spatial/number_scenarios_{CORPUS}.pdf")

### **Merged Publication Count: Number of Nodes, Time Steps and Scenarios**

The temporal panels (Number of Nodes, Number of Time Steps; Step 5) and the spatial panels (Number of Scenarios; above) are combined in one figure. A publication contributing to several panels is counted once. Run Step 5 with the same `FULL_TEXT_ONLY` setting first.

In [ ]:
panel_splits = {
    "Number of Nodes / Time Steps (Step 5)": load_text_availability_split(
        f"{CONTRIBUTING_DOIS_DIR}/number_nodes_timesteps_{CORPUS}.json"
    ),
    "Number of Scenarios (Step 6)": load_text_availability_split(
        f"{CONTRIBUTING_DOIS_DIR}/number_scenarios_{CORPUS}.json"
    ),
}
for name, split in panel_splits.items():
    print(f"{name}: {describe_corpus(CORPUS, split=split)}")

merged_split = merge_text_availability_splits(list(panel_splits.values()))
overlap = set.intersection(*[s["full_text"] | s["abstract_only"] for s in panel_splits.values()])
print(f"\nMerged figure: {describe_corpus(CORPUS, split=merged_split)}")
print(f"Publications contributing to both parts: {len(overlap):,}")